# 04 · End-to-end RAG evaluation

The project's thesis is *evaluation-first* RAG: every answer is tied to
retrieved evidence and scored with explicit metrics. This notebook runs the
full pipeline — ingest → retrieve → generate → evaluate — over the golden set
and exports a report, exactly as a CI regression job would.

In [1]:
import json
from pathlib import Path

import pandas as pd

from ragops_lab.evaluation import (
    evaluate_answer,
    export_evaluation_report_csv,
    export_evaluation_report_markdown,
)
from ragops_lab.generation import GenerationService, HeuristicLLMClient
from ragops_lab.ingestion import ChunkingConfig, ingest_directory, load_chunks_jsonl
from ragops_lab.retrieval import BM25Retriever

DATA = Path("../data")
ARTIFACTS = Path("../artifacts/evaluation")

## 1. Build the pipeline and load the golden questions

In [2]:
chunks_path = DATA / "processed" / "chunks.jsonl"
ingest_directory(DATA / "sample_documents", chunks_path, ChunkingConfig(chunk_size=220, overlap=20))
chunks = load_chunks_jsonl(chunks_path)
retriever = BM25Retriever(chunks)
service = GenerationService(HeuristicLLMClient())

golden = json.loads((DATA / "golden" / "qa.json").read_text())
golden

[{'query': 'Which Apollo mission first landed on the Moon?',
  'relevant_chunk_ids': ['apollo-program:0']},
 {'query': 'Which metrics are useful for RAG evaluation?',
  'relevant_chunk_ids': ['rag-evaluation:0']}]

## 2. Evaluate every golden question

For each question we retrieve context, generate a cited answer, and score it on
context precision/recall, answer relevance, faithfulness, and citation support.

In [3]:
records = []
detailed = []
for example in golden:
    question = example["query"]
    reference_ids = example["relevant_chunk_ids"]
    contexts = retriever.search(question, top_k=3)
    answer = service.answer(question, contexts, model_name="heuristic-local")
    result = evaluate_answer(
        question,
        answer,
        contexts,
        reference_chunk_ids=reference_ids,
    )
    detailed.append((question, answer, result))
    records.append(
        {
            "question": question[:42],
            "ctx_precision": round(result.context_precision, 2),
            "ctx_recall": round(result.context_recall, 2),
            "relevance": round(result.answer_relevance, 2),
            "faithfulness": round(result.faithfulness, 2),
            "citation": round(result.citation_support, 2),
            "unsupported": result.unsupported_claim_count,
        }
    )

frame = pd.DataFrame(records)
frame

,question,ctx_precision,ctx_recall,relevance,faithfulness,citation,unsupported
0,Which Apollo mission first landed on the M,1.0,1.0,0.62,1.0,1.0,0
1,Which metrics are useful for RAG evaluatio,0.5,1.0,0.29,1.0,1.0,0


## 3. Aggregate metrics across the dataset

In [4]:
numeric = frame.drop(columns=["question"])
numeric.mean().round(3).to_frame("mean")

,mean
ctx_precision,0.750
ctx_recall,1.000
relevance,0.455
faithfulness,1.000
citation,1.000
unsupported,0.000


## 4. Export a report artifact

The evaluation exporters write CSV and Markdown to `artifacts/evaluation/` —
the same artifacts a CI job would upload for inspection. We export the first
question's report as an example.

In [5]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
_, _, first_result = detailed[0]
export_evaluation_report_csv(first_result, ARTIFACTS / "report.csv")
export_evaluation_report_markdown(first_result, ARTIFACTS / "report.md")

print((ARTIFACTS / "report.md").read_text())

# Evaluation Report

- Context precision: 1.00
- Context recall: 1.00
- Answer relevance: 0.62
- Faithfulness: 1.00
- Citation support: 1.00
- Unsupported claims: 0
- Refusal correct: None


This closes the loop: ingestion, retrieval, grounded generation, and scored
evaluation all run on the same package code exposed through the CLI and API.
Pointing the retrievers and generator at real embeddings or a hosted model
changes only their construction — the evaluation harness shown here stays the
same.